# Discrete Random Variables
**Topic:** Random Variables & Distributions

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Define** a discrete random variable and its probability mass function (PMF)
- **Calculate** the expected value and variance of a discrete random variable
- **Interpret** the PMF as a complete description of a variable's probabilistic behavior

> **Tip:** Start by looking at the **PMF bar chart** before doing any math, notice that all the bars together sum to exactly 1, just like probabilities must.

---
## How we got here

In *01–03* we studied probability as a tool for single events. In *04–08* we described datasets we had already collected. Now we combine both ideas: a **random variable** is a variable whose value is the outcome of a random process, and the PMF is the probability's version of a histogram, showing how likely each outcome is before we collect any data.

---
## Why this matters for data science

Discrete random variables model the kinds of counts and categories that appear constantly in ML: the number of clicks on an ad, the number of defects in a product batch, the number of words in a document. Understanding expected value is essential for evaluating classifiers, the expected loss under a probability distribution is exactly what cross-entropy loss minimizes.

---
## Try it yourself

In [2]:
# ── Widget: Weighted Die & PMF Explorer ───────────────────────────────────────
out = Output()

p_sliders = [
    FloatSlider(
        value=0.20, min=0.0, max=1.0, step=0.05,
        description=f"P(X={k}):",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="400px"),
    )
    for k in range(1, 7)
]

preset_dropdown = Dropdown(
    options=[
        ("Fair Die", "fair"),
        ("Loaded High (favors 5–6)", "high"),
        ("Loaded Low (favors 1–2)", "low"),
        ("All-or-Nothing (only 1 and 6)", "all_or_nothing"),
    ],
    value="fair",
    description="Preset:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="400px"),
)

PRESETS = {
    "fair":           [0.20, 0.20, 0.20, 0.20, 0.20, 0.20],
    "high":           [0.05, 0.05, 0.10, 0.10, 0.35, 0.35],
    "low":            [0.35, 0.35, 0.10, 0.10, 0.05, 0.05],
    "all_or_nothing": [0.50, 0.00, 0.00, 0.00, 0.00, 0.50],
}

_state = {"updating": False}
_k = np.arange(1, 7)


def render():
    raw   = np.array([s.value for s in p_sliders], dtype=float)
    total = raw.sum()
    probs = raw / total if total > 0 else np.ones(6) / 6

    ev  = float(np.dot(_k, probs))
    var = float(np.dot((_k - ev) ** 2, probs))
    std = float(np.sqrt(var))

    if abs(ev - round(ev)) < 0.05:
        face   = int(round(ev))
        interp = f"On average you expect to roll {ev:.2f}, close to face {face} on the die."
    else:
        interp = (
            f"On average you expect to roll {ev:.2f} "
            f"— but this outcome never appears on any die face."
        )

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=_k,
        y=probs,
        marker_color=PALETTE["primary"],
        name="P(X = k)",
        text=[f"{p:.3f}" for p in probs],
        textposition="outside",
    ))
    fig.add_vline(
        x=ev,
        line_color=PALETTE["secondary"],
        line_width=2.5,
        line_dash="dash",
        annotation_text=f"E[X] = {ev:.2f}",
        annotation_position="top right",
    )

    plot_layout = base_layout(
        title="PMF: Weighted Die Roll — P(X = k)",
        xaxis_title="Face Value (k)",
        yaxis_title="P(X = k)",
    )
    plot_layout.update(
        xaxis=dict(tickmode="linear", dtick=1, range=[0.5, 6.5]),
        yaxis=dict(range=[0, min(1.0, float(probs.max()) + 0.15)]),
        showlegend=False,
    )
    fig.update_layout(plot_layout)

    html_panel = f"""
    <div style="font-family: sans-serif; background: #f8f9fa; border-radius: 8px;
                padding: 16px 24px; margin-top: 8px; line-height: 2; font-size: 14px;">
        <b>Summary Statistics</b>
        <table style="border-collapse: collapse; margin-top: 8px; width: 100%;">
            <tr>
                <td style="color: #555; padding-right: 24px;">E[X] = &Sigma; k &middot; P(X=k)</td>
                <td style="font-weight: bold; color: {PALETTE['primary']};">{ev:.4f}</td>
            </tr>
            <tr>
                <td style="color: #555;">Var(X) = &Sigma; (k &minus; E[X])&sup2; &middot; P(X=k)</td>
                <td style="font-weight: bold;">{var:.4f}</td>
            </tr>
            <tr>
                <td style="color: #555;">&sigma; = &radic;Var(X)</td>
                <td style="font-weight: bold;">{std:.4f}</td>
            </tr>
            <tr>
                <td style="color: #555;">&Sigma; P(X=k)</td>
                <td style="font-weight: bold; color: #2a9d8f;">1.0000 &#10003;</td>
            </tr>
        </table>
        <p style="margin-top: 12px; font-style: italic; color: #444;">{interp}</p>
    </div>
    """

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))
        display(HTML(html_panel))


def on_slider_change(change):
    if not _state["updating"]:
        render()


def on_preset_change(change):
    _state["updating"] = True
    for slider, w in zip(p_sliders, PRESETS[change["new"]]):
        slider.value = w
    _state["updating"] = False
    render()


for s in p_sliders:
    s.observe(on_slider_change, names="value")
preset_dropdown.observe(on_preset_change, names="value")

ui = VBox([preset_dropdown, *p_sliders, out])

if not hasattr(out, "_initialized"):
    display(ui)
    out._initialized = True
render()

---
## What's happening?

A **random variable** X maps each outcome of a random process to a number. When X can only take on a countable set of values (like the roll of a die), we call it **discrete**.

| Symbol | What it represents |
|--------|-------------------|
| X | The random variable itself (a rule, not a value) |
| x | A specific value X might take |
| P(X = x) | Probability mass function (PMF): probability X equals exactly x |
| E[X] | Expected value: the long-run average outcome |
| Var(X) | Variance: expected squared distance from E[X] |

$$E[X] = \sum_{x} x \cdot P(X = x)$$

$$\text{Var}(X) = E[(X - E[X])^2] = \sum_{x} (x - E[X])^2 \cdot P(X = x)$$

### PMF rules: two things that must always be true

Every valid PMF satisfies: (1) P(X = x) ≥ 0 for all x, and (2) the sum of all probabilities equals exactly 1. Go back to the widget and verify this, no matter how you adjust the weights, the bars should always total 1.

---
## Real-world example: Customer service call volume

A call center tracks how many customer service calls arrive per 10-minute window. This is discrete, you can't have 2.7 calls. The PMF tells the center how to staff: if E[X] = 4 calls per window, they need enough agents to handle that average plus some buffer for variance.

The chart below shows a realistic PMF for calls per 10-minute window, derived from a Poisson distribution with λ = 4 (a typical small call center rate). Notice:

- **Notice:** The most likely outcome (mode) is 3 or 4 calls, not exactly E[X] = 4, the expected value is a long-run average, not the most probable single outcome
- **Notice:** There is nonzero probability of 0 calls and of 10+ calls, the tails extend in both directions
- **Notice:** The distribution is slightly right-skewed, occasionally the center gets slammed with far more calls than typical, but rarely gets far below average

> **Discussion question:** If the call center staffs for exactly E[X] = 4 calls per window, what fraction of 10-minute windows will they be understaffed (more calls than agents)? How does Var(X) help answer this question?

In [3]:
# ── Poisson PMF for call center arrivals ─────────────────────────────────────
lam   = 4   # average calls per 10-minute window (realistic for a small center)
k_max = 14
k_vals   = np.arange(0, k_max + 1)
pmf_vals = stats.poisson.pmf(k_vals, lam)
ev       = lam  # E[X] = lambda for Poisson

fig = go.Figure()
fig.add_trace(go.Bar(
    x=k_vals, y=pmf_vals,
    marker_color=PALETTE["primary"],
    name="P(X = k)",
    text=[f"{p:.3f}" for p in pmf_vals],
    textposition="outside",
))
fig.add_vline(x=ev, line_color=PALETTE["secondary"], line_width=2.5, line_dash="dash",
              annotation_text=f"E[X] = {ev}", annotation_position="top right")
layout = base_layout(
    title=f"PMF: Calls per 10-Minute Window  (λ = {lam})",
    xaxis_title="Number of Calls (k)",
    yaxis_title="P(X = k)",
)
layout.update(xaxis=dict(tickmode="linear", dtick=1))
fig.update_layout(layout)
fig.show()

### Common discrete random variables in data science

| Random variable | Values X can take | What E[X] represents |
|----------------|-------------------|---------------------|
| Bernoulli(p) | {0, 1} | Probability of success on one trial |
| Binomial(n, p) | {0, 1, …, n} | Expected number of successes in n trials |
| Poisson(λ) | {0, 1, 2, …} | Expected number of events per interval |
| Geometric(p) | {1, 2, 3, …} | Expected number of trials until first success |
| Negative Binomial | {r, r+1, …} | Expected trials until r-th success |

---
## Key takeaway

> **A discrete random variable's PMF assigns a probability to every possible outcome; the expected value is the probability-weighted average — the outcome you would predict if forced to guess before seeing the data.**

---
*Next up: Discrete Distributions — the most important named discrete distributions and when each one applies*